#INITIAL SETUP

In [ ]:
pip install nlpaug

In [ ]:
pip install datasets

In [ ]:
import torch
from torch import nn
from torch.utils.data import  DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from transformers import RobertaTokenizer, RobertaModel
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
import re
import torch.nn.functional as F
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score
from transformers import BertTokenizer, BertModel
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
from datasets import load_dataset
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support, precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import jaccard_score
from collections import Counter
import random

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)

set_seed(42)

In [ ]:

# Check GPU availability
print("Is CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load the dataset
ds = load_dataset("go_emotions", "simplified")
# Get emotion names
emotion_names = ds['train'].features['labels'].feature.names
num_labels = len(emotion_names)
threshold = 0.4


# Analysis

In [ ]:


# Function to count emotion frequencies
def count_emotions(split):
    emotion_counts = {emotion: 0 for emotion in emotion_names}
    for labels in split['labels']:
        for label in labels:
            emotion_counts[emotion_names[label]] += 1
    return emotion_counts
# Count emotions for each split
train_counts = count_emotions(ds['train'])
val_counts = count_emotions(ds['validation'])
test_counts = count_emotions(ds['test'])

# Combine counts
total_counts = {emotion: train_counts[emotion] + val_counts[emotion] + test_counts[emotion]
                for emotion in emotion_names}

# Create a DataFrame for easier manipulation
df_counts = pd.DataFrame.from_dict(total_counts, orient='index', columns=['count'])
df_counts = df_counts.sort_values('count', ascending=False)

In [ ]:
color_palette = sns.color_palette("husl", n_colors=len(df_counts))

# Plotting
plt.figure(figsize=(20, 10))
sns.barplot(x=df_counts.index, y='count', data=df_counts)
plt.title('Distribution of Emotions', fontsize=20)
plt.xlabel('Emotions', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:

df_train = pd.DataFrame.from_dict(train_counts, orient='index', columns=['count']).reset_index().rename(columns={'index': 'emotion'})
df_val = pd.DataFrame.from_dict(val_counts, orient='index', columns=['count']).reset_index().rename(columns={'index': 'emotion'})
df_test = pd.DataFrame.from_dict(test_counts, orient='index', columns=['count']).reset_index().rename(columns={'index': 'emotion'})

# Function to plot distribution for a dataset
def plot_distribution(df, title):
    plt.figure(figsize=(20, 10))
    sns.barplot(x='emotion', y='count', data=df.sort_values('count', ascending=False))
    plt.title(f'Distribution of Emotions in {title} Dataset', fontsize=20)
    plt.xlabel('Emotions', fontsize=14)
    plt.ylabel('Count', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# Plot distributions for each dataset
plot_distribution(df_train, 'Train')
plot_distribution(df_val, 'Validation')
plot_distribution(df_test, 'Test')


In [ ]:
def calculate_jaccard_similarity(split):
    # Create matrices to store co-occurrences and individual occurrences
    cooccur = np.zeros((len(emotion_names), len(emotion_names)))
    occur = np.zeros(len(emotion_names))

    # Count co-occurrences and individual occurrences
    for labels in split['labels']:
        for i in labels:
            occur[i] += 1
            for j in labels:
                cooccur[i][j] += 1

    # Calculate Jaccard similarity
    jaccard = np.zeros((len(emotion_names), len(emotion_names)))
    for i in range(len(emotion_names)):
        for j in range(len(emotion_names)):
            intersection = cooccur[i][j]
            union = occur[i] + occur[j] - intersection
            jaccard[i][j] = intersection / union if union > 0 else 0

    return pd.DataFrame(jaccard, index=emotion_names, columns=emotion_names)

def get_top_cooccurrences(jaccard_df, n=15):
    # Create a mask to keep only the upper triangle of the matrix
    mask = np.triu(np.ones(jaccard_df.shape), k=1).astype(bool)

    # Apply the mask to get only the upper triangle values
    upper_tri = jaccard_df.where(mask)

    # Stack the DataFrame to get emotion pairs and their similarities
    stacked = upper_tri.stack().reset_index()
    stacked.columns = ['emotion1', 'emotion2', 'similarity']

    # Sort by similarity in descending order and get top n
    return stacked.sort_values('similarity', ascending=False).head(n)

# Calculate Jaccard similarity for each split
train_jaccard = calculate_jaccard_similarity(ds['train'])
val_jaccard = calculate_jaccard_similarity(ds['validation'])
test_jaccard = calculate_jaccard_similarity(ds['test'])

# Get top 10 co-occurrences for each split
print("Top 15 co-occurring emotions (Jaccard similarity) - Train set:")
print(get_top_cooccurrences(train_jaccard))

print("\nTop 15 co-occurring emotions (Jaccard similarity) - Validation set:")
print(get_top_cooccurrences(val_jaccard))

print("\nTop 15 co-occurring emotions (Jaccard similarity) - Test set:")
print(get_top_cooccurrences(test_jaccard))

In [ ]:

def calculate_combined_jaccard_similarity(splits):
    cooccur = np.zeros((len(emotion_names), len(emotion_names)))
    occur = np.zeros(len(emotion_names))
    for split in splits:
        for labels in split['labels']:
            for i in labels:
                occur[i] += 1
                for j in labels:
                    cooccur[i][j] += 1
    jaccard = np.zeros((len(emotion_names), len(emotion_names)))
    for i in range(len(emotion_names)):
        for j in range(len(emotion_names)):
            intersection = cooccur[i][j]
            union = occur[i] + occur[j] - intersection
            jaccard[i][j] = intersection / union if union > 0 else 0

    return pd.DataFrame(jaccard, index=emotion_names, columns=emotion_names)

combined_jaccard = calculate_combined_jaccard_similarity([ds['train'], ds['validation'], ds['test']])

top_combined = get_top_cooccurrences(combined_jaccard, n=15)

formatted_combined = top_combined.copy()
formatted_combined['similarity'] = formatted_combined['similarity'].apply(lambda x: f"{x:.6f}")

print("Top 15 co-occurring emotions (Jaccard similarity across all sets combined):")
print(formatted_combined.to_string(index=False))

In [ ]:
def check_duplicates(df):
    # Find exact duplicates (same ID, text, and labels)
    exact_duplicates = df[df.duplicated(keep=False)]
    print(f"\nNumber of exact duplicate entries: {len(exact_duplicates)}")

    # Find entries with the same text and labels but different IDs
    content_duplicates = df.groupby(['text', 'labels']).filter(lambda x: len(x) > 1 and len(x['id'].unique()) > 1)
    print(f"\nNumber of entries with same content but different IDs: {len(content_duplicates)}")

    # Return the duplicates for further analysis if needed
    return exact_duplicates, content_duplicates

In [ ]:
def combine_splits(ds, split_names):
    combined_texts = []
    for split_name in split_names:
        combined_texts.extend(ds[split_name]['text'])
    return pd.Series(combined_texts)

def check_split(split_name):
    print(f"\nChecking {split_name} split:")
    split = ds[split_name]

    df = pd.DataFrame({
        'id': split['id'],
        'text': split['text'],
        'labels': [tuple(labels) for labels in split['labels']]  # Convert lists to tuples
    })

    print("Missing values:")
    print(df.isnull().sum())

    print(f"Empty text entries: {df['text'].str.strip().eq('').sum()}")
    print(f"Entries with no labels: {sum(len(labels) == 0 for labels in df['labels'])}")
    print(f"Entries with invalid label indices: {sum(any(label >= len(emotion_names) for label in labels) for labels in df['labels'])}")

    exact_duplicates, content_duplicates = check_duplicates(df)

    all_labels = [label for labels in df['labels'] for label in labels]
    print("\nLabel distribution:")
    for emotion_id, count in Counter(all_labels).most_common():
        print(f"{emotion_names[emotion_id]}: {count}")

    text_lengths = df['text'].str.len()
    print(f"\nText length statistics:\n{text_lengths.describe()}")



for split in ['train', 'validation', 'test']:
    check_split(split)


# Combine all splits
combined_texts = combine_splits(ds, ['train', 'validation', 'test'])

text_lengths = combined_texts.str.len()

print("\nText length statistics for combined splits:")
print(text_lengths.describe())

all_labels = [label for split in ['train', 'validation', 'test'] for labels in ds[split]['labels'] for label in labels]
print("\nOverall label distribution:")
for emotion_id, count in Counter(all_labels).most_common():
    print(f"{emotion_names[emotion_id]}: {count}")



In [ ]:
def get_id_text_label_tuples(split):
    return set(zip(split['id'], split['text'], map(tuple, split['labels'])))

train_tuples = get_id_text_label_tuples(ds['train'])
val_tuples = get_id_text_label_tuples(ds['validation'])
test_tuples = get_id_text_label_tuples(ds['test'])

# Check for exact duplicates (same ID, text, and labels)
train_val_exact_overlap = len(train_tuples.intersection(val_tuples))
train_test_exact_overlap = len(train_tuples.intersection(test_tuples))
val_test_exact_overlap = len(val_tuples.intersection(test_tuples))

print(f"Exact overlap (ID, text, and labels) between train and validation sets: {train_val_exact_overlap}")
print(f"Exact overlap (ID, text, and labels) between train and test sets: {train_test_exact_overlap}")
print(f"Exact overlap (ID, text, and labels) between validation and test sets: {val_test_exact_overlap}")

# Check for text and label overlap (ignoring ID)
def get_text_label_pairs(tuples):
    return set((text, labels) for _, text, labels in tuples)

train_pairs = get_text_label_pairs(train_tuples)
val_pairs = get_text_label_pairs(val_tuples)
test_pairs = get_text_label_pairs(test_tuples)

train_val_content_overlap = len(train_pairs.intersection(val_pairs))
train_test_content_overlap = len(train_pairs.intersection(test_pairs))
val_test_content_overlap = len(val_pairs.intersection(test_pairs))

print(f"\nContent overlap (text and labels, different ID) between train and validation sets: {train_val_content_overlap}")
print(f"Content overlap (text and labels, different ID) between train and test sets: {train_test_content_overlap}")
print(f"Content overlap (text and labels, different ID) between validation and test sets: {val_test_content_overlap}")


def get_text_to_id_label_dict(tuples):
    text_dict = {}
    for id_, text, labels in tuples:
        if text not in text_dict:
            text_dict[text] = set()
        text_dict[text].add((id_, labels))
    return text_dict

train_dict = get_text_to_id_label_dict(train_tuples)
val_dict = get_text_to_id_label_dict(val_tuples)
test_dict = get_text_to_id_label_dict(test_tuples)

def count_text_only_overlap(dict1, dict2):
    count = 0
    for text in dict1.keys() & dict2.keys():  # Texts that appear in both sets
        if dict1[text] != dict2[text]:  # Check if ID and labels are different
            count += 1
    return count

train_val_text_only_overlap = count_text_only_overlap(train_dict, val_dict)
train_test_text_only_overlap = count_text_only_overlap(train_dict, test_dict)
val_test_text_only_overlap = count_text_only_overlap(val_dict, test_dict)

print(f"\nText-only overlap (same text, different ID and labels) between train and validation sets: {train_val_text_only_overlap}")
print(f"Text-only overlap (same text, different ID and labels) between train and test sets: {train_test_text_only_overlap}")
print(f"Text-only overlap (same text, different ID and labels) between validation and test sets: {val_test_text_only_overlap}")

In [ ]:
def get_text_label_pairs(split):
    return {(text, tuple(sorted(labels))) for text, labels in zip(split['text'], split['labels'])}

train_pairs = get_text_label_pairs(ds['train'])
val_pairs = get_text_label_pairs(ds['validation'])
test_pairs = get_text_label_pairs(ds['test'])

def find_different_labels(split1_pairs, split2_pairs, split1_name, split2_name):
    split1_texts = {text for text, _ in split1_pairs}
    split2_texts = {text for text, _ in split2_pairs}

    common_texts = split1_texts.intersection(split2_texts)

    different_labels = []
    for text in common_texts:
        labels1 = [labels for t, labels in split1_pairs if t == text]
        labels2 = [labels for t, labels in split2_pairs if t == text]

        if labels1 and labels2:  # Make sure we have labels for both
            if set(labels1[0]) != set(labels2[0]):
                different_labels.append((text, labels1[0], labels2[0]))

    print(f"\nTexts with different labels between {split1_name} and {split2_name}:")
    for text, labels1, labels2 in different_labels:
        print(f"Text: {text}")
        print(f"{split1_name} labels: {[emotion_names[l] for l in labels1]}")
        print(f"{split2_name} labels: {[emotion_names[l] for l in labels2]}")
        print("---")

    return different_labels



train_val_diff = find_different_labels(train_pairs, val_pairs, "Train", "Validation")
train_test_diff = find_different_labels(train_pairs, test_pairs, "Train", "Test")
val_test_diff = find_different_labels(val_pairs, test_pairs, "Validation", "Test")

print(f"\nTotal cases of different labels:")
print(f"Train-Validation: {len(train_val_diff)}")
print(f"Train-Test: {len(train_test_diff)}")
print(f"Validation-Test: {len(val_test_diff)}")

In [ ]:
def count_special_tokens(texts):
    pattern = r'\[(NAME|RELIGION)\]'
    token_counter = Counter()

    for text in texts:
        matches = re.findall(pattern, text)
        token_counter.update(matches)

    return token_counter

def analyze_special_tokens(split_name):
    texts = ds[split_name]['text']
    token_counts = count_special_tokens(texts)

    print(f"\nSpecial tokens in {split_name} split:")
    for token, count in token_counts.most_common():
        print(f"[{token}]: {count}")

    total_tokens = sum(token_counts.values())
    unique_tokens = len(token_counts)
    print(f"\nTotal special tokens: {total_tokens}")


# Analyze each split
for split in ['train', 'validation', 'test']:
    analyze_special_tokens(split)

# Analyze all splits combined
all_texts = ds['train']['text'] + ds['validation']['text'] + ds['test']['text']
all_token_counts = count_special_tokens(all_texts)

print("\nSpecial tokens in all splits combined:")
for token, count in all_token_counts.most_common():
    print(f"[{token}]: {count}")

total_tokens = sum(all_token_counts.values())
unique_tokens = len(all_token_counts)
print(f"\nTotal special tokens across all splits: {total_tokens}")
print(f"Unique special tokens across all splits: {unique_tokens}")

#SETUP CLASSES / FUNCTIONS

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.epsilon = 1e-8  # Small constant for numerical stability

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) + self.epsilon
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    return total_loss / len(dataloader), all_probs, all_labels

In [ ]:
def train(model, dataloader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
def calculate_metrics(probs, labels, threshold=0.4):
    preds = (probs > threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)

    report = classification_report(labels, preds, target_names=emotion_names, zero_division=0)

    macro = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    micro = precision_recall_fscore_support(labels, preds, average='micro', zero_division=0)

    return {
        'weighted_precision': precision,
        'weighted_recall': recall,
        'weighted_f1': f1,
        'classification_report': report,
        'macro_precision': macro[0],
        'macro_recall': macro[1],
        'macro_f1': macro[2],
        'micro_precision': micro[0],
        'micro_recall': micro[1],
        'micro_f1': micro[2]
    }

In [ ]:

def evaluate_and_print(model, dataloader, criterion, device, model_name, threshold=0.4):
    loss, probs, labels = evaluate(model, dataloader, criterion, device)
    preds = (probs > threshold).astype(int)
    metrics = calculate_metrics(probs, labels, threshold=threshold)

    print(f"\n{model_name} Results (Threshold: {threshold}):")
    print(f"Loss: {loss:.4f}")
    print(f"Weighted Precision: {metrics['weighted_precision']:.4f}")
    print(f"Weighted Recall: {metrics['weighted_recall']:.4f}")
    print(f"Weighted F1-score: {metrics['weighted_f1']:.4f}")

    print("\nMacro-averaged metrics:")
    print(f"Precision: {metrics['macro_precision']:.4f}")
    print(f"Recall: {metrics['macro_recall']:.4f}")
    print(f"F1-score: {metrics['macro_f1']:.4f}")
    print("\nMicro-averaged metrics:")
    print(f"Precision: {metrics['micro_precision']:.4f}")
    print(f"Recall: {metrics['micro_recall']:.4f}")
    print(f"F1-score: {metrics['micro_f1']:.4f}")

    print("\nClassification Report:")
    print(metrics['classification_report'])




    precision, recall, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)

    print("\nDetailed per-class metrics:")
    for i, emotion in enumerate(emotion_names):
        print(f"{emotion}:")
        print(f"  Precision: {precision[i]:.4f}")
        print(f"  Recall: {recall[i]:.4f}")
        print(f"  F1-score: {f1[i]:.4f}")
        print(f"  Support: {support[i]}")


    metrics['loss'] = loss
    metrics['probs'] = probs
    metrics['labels'] = labels
    metrics['preds'] = preds
    metrics['threshold'] = threshold
    metrics['per_class_precision'] = precision
    metrics['per_class_recall'] = recall
    metrics['per_class_f1'] = f1
    metrics['per_class_support'] = support

    return metrics

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = [int(label) for label in self.labels[idx]]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        label_vector = torch.zeros(len(emotion_names))
        label_vector[labels] = 1

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': label_vector
        }

In [ ]:

from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, auc

def plot_performance_curves(y_true, y_pred, model_name):
    # Compute micro-average Precision-Recall curve
    precision, recall, _ = precision_recall_curve(y_true.ravel(), y_pred.ravel())
    avg_precision = average_precision_score(y_true, y_pred, average="micro")

    # Compute micro-average ROC curve
    fpr, tpr, _ = roc_curve(y_true.ravel(), y_pred.ravel())
    roc_auc = auc(fpr, tpr)


    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(recall, precision, color='b', lw=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} Micro-average Precision-Recall curve\nAverage Precision = {avg_precision:.2f}')


    plt.subplot(1, 2, 2)
    plt.plot(fpr, tpr, color='b', lw=2)
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} Micro-average ROC curve\nAUC = {roc_auc:.2f}')

    plt.tight_layout()
    plt.savefig(f'{model_name}_performance_curves.png')
    plt.show()
    plt.close()


In [ ]:
def hyperparameter_tuning_plots(results,model_name,num_epochs):
    plt.figure(figsize=(12, 6))
    for result in results:
        label = f"gamma={result['gamma']}, lr={result['lr']}"
        plt.plot(range(1, num_epochs+1), result['losses'], label=label)
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title(f'{model_name} Training Loss During Hyperparameter Tuning')
    plt.legend()
    plt.savefig(f'{model_name}_hyperparameter_tuning_loss.png')
    plt.show()
    plt.close()

    plt.figure(figsize=(12, 6))
    for result in results:
        label = f"gamma={result['gamma']}, lr={result['lr']}"
        plt.plot(range(1, num_epochs+1), result['f1_scores'], label=label)
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.title(f'{model_name} F1 Score During Hyperparameter Tuning')
    plt.legend()
    plt.savefig(f'{model_name}_hyperparameter_tuning_f1.png')
    plt.show()
    plt.close()

In [ ]:
def final_model_plots(total_epochs,best_epoch,train_losses,val_losses,val_f1_scores,model_name):
  plt.figure(figsize=(12, 6))
  plt.plot(range(1, total_epochs + 1), train_losses, label='Training Loss')
  plt.plot(range(1, total_epochs + 1), val_losses, label='Validation Loss')
  plt.axvline(x=best_epoch, color='r', linestyle='--', label='Best Epoch During Hyperparameter Tuning')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.title(f'{model_name} Training and Validation Loss')
  plt.legend()
  plt.savefig(f'{model_name}_training_validation_loss.png')
  plt.show()
  plt.close()

  # Plot validation F1 score
  plt.figure(figsize=(12, 6))
  plt.plot(range(1, total_epochs + 1), val_f1_scores, label='F1 Score')
  plt.axvline(x=best_epoch, color='r', linestyle='--', label='Best Epoch During Hyperparameter Tuning')
  plt.xlabel('Epoch')
  plt.ylabel('F1 Score')
  plt.title(f'{model_name} Validation F1 Score')
  plt.legend()
  plt.savefig(f'{model_name}_validation_f1_score.png')
  plt.show()
  plt.close()

#Oversamling

In [ ]:
def ml_ros(texts, labels, emotion_names, sampling_strategy=0.5, max_ratio=3, random_seed=42):
    random.seed(random_seed)

    label_counts = Counter()
    for label_list in labels:
        label_counts.update(label_list)

    median_count = sorted(label_counts.values())[len(label_counts) // 2]

    oversampled_texts = texts.copy()
    oversampled_labels = labels.copy()

    for label in range(len(emotion_names)):
        current_count = label_counts[label]
        if current_count < median_count:
            target_count = min(int(median_count * sampling_strategy), current_count * max_ratio)
            n_to_sample = max(0, int(target_count - current_count))

            if n_to_sample > 0:
                indices = [i for i, l in enumerate(labels) if label in l]
                sampled_indices = random.choices(indices, k=n_to_sample)

                for idx in sampled_indices:
                    oversampled_texts.append(texts[idx])
                    oversampled_labels.append(labels[idx].copy())

    return oversampled_texts, oversampled_labels



In [ ]:
def print_detailed_distribution(labels, emotion_names):
    label_counts = Counter()
    combo_counts = Counter()
    label_per_sample = []

    for label_list in labels:
        label_counts.update(label_list)
        combo_counts[tuple(sorted(label_list))] += 1
        label_per_sample.append(len(label_list))

    total_samples = len(labels)
    total_labels = sum(label_counts.values())
    multi_label_samples = sum(1 for count in label_per_sample if count > 1)

    print(f"Dataset Summary:")
    print(f"Total samples: {total_samples}")
    print(f"Total label occurrences: {total_labels}")
    print(f"Average labels per sample: {total_labels / total_samples:.2f}")
    print(f"Samples with multiple labels: {multi_label_samples} ({multi_label_samples/total_samples:.2%})")
    print(f"Unique label combinations: {len(combo_counts)}")

    print("\nLabel Distribution:")
    for emotion, count in label_counts.most_common():
        print(f"{emotion_names[emotion]:<15} {count:5d} ({count/total_samples:.2%})")


    print("\nLabel Count Distribution:")
    for count in range(1, max(label_per_sample) + 1):
        samples = sum(1 for c in label_per_sample if c == count)
        print(f"{count} label: {samples:5d} samples ({samples/total_samples:.2%})")


oversampled_texts, oversampled_labels = ml_ros(ds['train']['text'], ds['train']['labels'],emotion_names, sampling_strategy=0.5, max_ratio=3)
print("Original dataset distribution:")
print_detailed_distribution(ds['train']['labels'], emotion_names)

print("\nOversampled dataset distribution:")
print_detailed_distribution(oversampled_labels, emotion_names)

#ROBERTA

In [ ]:
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
special_tokens_dict = {'additional_special_tokens': ['[NAME]', '[RELIGION]']}
num_added_toks = roberta_tokenizer.add_special_tokens(special_tokens_dict)
print(f"We have added {num_added_toks} tokens")

In [ ]:

class RoBERTaForMultiLabelClassification(nn.Module):
    def __init__(self, num_labels, dropout_rate=0.3):
        super(RoBERTaForMultiLabelClassification, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

In [ ]:


train_dataset_roberta = EmotionDataset(oversampled_texts, oversampled_labels, roberta_tokenizer, max_len=128)
val_dataset_roberta = EmotionDataset(ds['validation']['text'], ds['validation']['labels'], roberta_tokenizer, max_len=128)
test_dataset_roberta = EmotionDataset(ds['test']['text'], ds['test']['labels'], roberta_tokenizer, max_len=128)

print(f"Train dataset size: {len(train_dataset_roberta)}")
print(f"Validation dataset size: {len(val_dataset_roberta)}")
print(f"Test dataset size: {len(test_dataset_roberta)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

train_loader = DataLoader(train_dataset_roberta, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset_roberta, batch_size=16)
test_loader = DataLoader(test_dataset_roberta, batch_size=16)


def hyperparameter_tuning_roberta():
    gammas = [2, 3]
    learning_rates = [5e-5, 1e-5, 7e-6]
    num_epochs = 3
    dropout_rate = 0.5
    weight_decay = 0.1

    best_params = None
    best_val_f1 = 0
    best_model_state = None
    best_history = None
    best_epoch = 0

    results = []

    for gamma in gammas:
        for lr in learning_rates:
            print(f"Training with gamma={gamma}, lr={lr}")
            roberta_model = RoBERTaForMultiLabelClassification(num_labels, dropout_rate=dropout_rate).to(device)
            roberta_model.roberta.resize_token_embeddings(len(roberta_tokenizer))

            optimizer = torch.optim.AdamW(roberta_model.parameters(), lr=lr,weight_decay=weight_decay)
            criterion = FocalLoss(gamma=gamma)
            scaler = GradScaler()

            train_losses = []
            val_losses = []
            val_f1_scores = []

            for epoch in range(num_epochs):
                train_loss = train(roberta_model, train_loader, optimizer, criterion, device, scaler)
                val_loss, val_probs, val_labels = evaluate(roberta_model, val_loader, criterion, device)
                val_preds = (val_probs > threshold).astype(int)
                val_f1 = f1_score(val_labels, val_preds, average='weighted')
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                val_f1_scores.append(val_f1)

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_params = {'gamma': gamma, 'lr': lr}
                    best_model_state = roberta_model.state_dict()
                    best_history = {
                        'train_losses': train_losses.copy(),
                        'val_losses': val_losses.copy(),
                        'val_f1_scores': val_f1_scores.copy()
                    }
                    best_epoch = epoch + 1
            results.append({
                'gamma': gamma,
                'lr': lr,
                'losses': train_losses,
                'val_losses': val_losses,
                'f1_scores': val_f1_scores
            })




    print(f"Best parameters: {best_params}")
    print(f"Best F1 score: {best_val_f1}")
    print(f"Best epoch: {best_epoch}")

    hyperparameter_tuning_plots(results,"RoBERTa",num_epochs)

    torch.save(best_model_state, 'roberta_best_model.pth')

    return best_params, best_model_state, best_history, best_epoch

# Hyperparameter tuning
best_params, best_model_state, best_history, best_epoch  = hyperparameter_tuning_roberta()

In [ ]:


# Train the final model with the best parameters
roberta_model = RoBERTaForMultiLabelClassification(num_labels, dropout_rate=0.5).to(device)
roberta_model.roberta.resize_token_embeddings(len(roberta_tokenizer))
roberta_model.load_state_dict(best_model_state)


optimizer = torch.optim.AdamW(roberta_model.parameters(), lr=best_params['lr'], weight_decay=0.1)
criterion = FocalLoss(gamma=best_params['gamma'])
scaler = GradScaler()
print("Further model training of the best model found in hyperparameter tuning")


num_epochs = 5
patience = 2
best_val_f1 = max(best_history['val_f1_scores'])
counter = 0

# Initialize lists to store metrics
train_losses = best_history['train_losses']
val_losses = best_history['val_losses']
val_f1_scores = best_history['val_f1_scores']




for epoch in range(num_epochs):
    train_loss = train(roberta_model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_probs, val_labels = evaluate(roberta_model, val_loader, criterion, device)
    val_preds = (val_probs > threshold).astype(int)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')

    # Append metrics for plotting
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1_scores.append(val_f1)

    print(f"Epoch {best_epoch + epoch + 1}/{best_epoch + num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation F1: {val_f1:.4f}")

    # Check if this is the best model so far
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(roberta_model.state_dict(), 'roberta_best_model.pth')
        print(f"New best model saved with F1 score: {best_val_f1:.4f}")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {best_epoch + epoch + 1}")
            break


In [ ]:
roberta_model.load_state_dict(torch.load('roberta_best_model.pth'))
total_epochs = best_epoch + epoch + 1
final_model_plots(total_epochs,best_epoch,train_losses,val_losses,val_f1_scores,'RoBERTa')


In [ ]:
roberta_metrics = evaluate_and_print(roberta_model, test_loader, criterion, device, "RoBERTa", 0.4)

In [ ]:

plot_performance_curves(roberta_metrics['labels'], roberta_metrics['preds'],"RoBERTa")

#XLNET

In [ ]:
from transformers import XLNetTokenizer, XLNetModel

In [ ]:
# Initialize tokenizer with special tokens
xlnet_tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')
special_tokens_dict = {'additional_special_tokens': ['[NAME]', '[RELIGION]']}
num_added_toks = xlnet_tokenizer.add_special_tokens(special_tokens_dict)

print(f"We have added {num_added_toks} tokens")

In [ ]:
class XLNetForMultiLabelClassification(nn.Module):
    def __init__(self, num_labels, dropout_rate=0.3):
        super(XLNetForMultiLabelClassification, self).__init__()
        self.xlnet = XLNetModel.from_pretrained('xlnet-base-cased')
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.xlnet.config.d_model, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.xlnet(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

In [ ]:
train_dataset_xlnet = EmotionDataset(oversampled_texts, oversampled_labels, xlnet_tokenizer, max_len=128)
val_dataset_xlnet = EmotionDataset(ds['validation']['text'], ds['validation']['labels'], xlnet_tokenizer, max_len=128)
test_dataset_xlnet = EmotionDataset(ds['test']['text'], ds['test']['labels'], xlnet_tokenizer, max_len=128)

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset_xlnet, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset_xlnet, batch_size=16)
test_loader = DataLoader(test_dataset_xlnet, batch_size=16)

In [ ]:
def hyperparameter_tuning_xlnet():
    gammas = [2, 3]
    learning_rates = [5e-5, 1e-5, 7e-6]
    num_epochs = 3
    dropout_rate = 0.5
    weight_decay = 0.1


    best_params = None
    best_val_f1 = 0
    best_model_state = None
    best_history = None
    best_epoch = 0

    results = []

    for gamma in gammas:
        for lr in learning_rates:
            print(f"Training XLNet with gamma={gamma}, lr={lr}")
            xlnet_model = XLNetForMultiLabelClassification(num_labels,dropout_rate=dropout_rate).to(device)
            xlnet_model.xlnet.resize_token_embeddings(len(xlnet_tokenizer))

            optimizer = torch.optim.AdamW(xlnet_model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = FocalLoss(gamma=gamma)
            scaler = GradScaler()

            train_losses = []
            val_losses = []
            val_f1_scores = []

            for epoch in range(num_epochs):
                train_loss = train(xlnet_model, train_loader, optimizer, criterion, device, scaler)
                val_loss, val_probs, val_labels = evaluate(xlnet_model, val_loader, criterion, device)
                val_preds = (val_probs > threshold).astype(int)
                val_f1 = f1_score(val_labels, val_preds, average='weighted')

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                val_f1_scores.append(val_f1)
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_params = {'gamma': gamma, 'lr': lr}
                    best_model_state = xlnet_model.state_dict()
                    best_history = {
                        'train_losses': train_losses.copy(),
                        'val_losses': val_losses.copy(),
                        'val_f1_scores': val_f1_scores.copy()
                    }
                    best_epoch = epoch + 1

            results.append({
                'gamma': gamma,
                'lr': lr,
                'losses': train_losses,
                'val_losses': val_losses,
                'f1_scores': val_f1_scores
            })



    print(f"Best parameters: {best_params}")
    print(f"Best F1 score: {best_val_f1}")
    print(f"Best epoch: {best_epoch}")

    hyperparameter_tuning_plots(results,"XLNet",num_epochs)

    torch.save(best_model_state, 'xlnet_best_model.pth')

    return best_params, best_model_state, best_history, best_epoch

# Run the hyperparameter tuning
print("Running XLNet hyperparameter tuning...")

best_params, best_model_state,best_history,best_epoch = hyperparameter_tuning_xlnet()


In [ ]:
from transformers import get_linear_schedule_with_warmup


xlnet_model = XLNetForMultiLabelClassification(num_labels, dropout_rate=0.5).to(device)
xlnet_model.xlnet.resize_token_embeddings(len(xlnet_tokenizer))
xlnet_model.load_state_dict(best_model_state)  # Load the best model state

optimizer = torch.optim.AdamW(xlnet_model.parameters(), lr=best_params['lr'], weight_decay=0.1)
criterion = FocalLoss(gamma=best_params['gamma'])
scaler = GradScaler()
print("Further model training of the best model found in hyperparameter tuning")
num_epochs = 5
patience = 2
best_val_f1 = max(best_history['val_f1_scores'])
counter = 0

# Initialize lists to store metrics
train_losses = best_history['train_losses']
val_losses = best_history['val_losses']
val_f1_scores = best_history['val_f1_scores']

for epoch in range(num_epochs):
    train_loss = train(xlnet_model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_probs, val_labels = evaluate(xlnet_model, val_loader, criterion, device)
    val_preds = (val_probs > threshold).astype(int)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1_scores.append(val_f1)

    print(f"Epoch {best_epoch + epoch + 1}/{best_epoch + num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(xlnet_model.state_dict(), 'xlnet_best_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience and epoch + 1 < num_epochs :
                print(f"Early stopping triggered at epoch {best_epoch + epoch + 1}")
                break



In [ ]:

total_epochs = best_epoch + epoch + 1
final_model_plots(total_epochs,best_epoch,train_losses,val_losses,val_f1_scores,'XLNet')


In [ ]:
xlnet_model.load_state_dict(torch.load('xlnet_best_model.pth'))
xlnet_metrics = evaluate_and_print(xlnet_model, test_loader, criterion, device, "XLNet", threshold = 0.4)

In [ ]:

plot_performance_curves(xlnet_metrics['labels'],xlnet_metrics['preds'],"XLNet")

#ELECTRA

In [ ]:
from transformers import ElectraTokenizer, ElectraModel

# Initialize tokenizer with special tokens
electra_tokenizer = ElectraTokenizer.from_pretrained('google/electra-base-discriminator')
special_tokens_dict = {'additional_special_tokens': ['[NAME]', '[RELIGION]']}
num_added_toks = electra_tokenizer.add_special_tokens(special_tokens_dict)

print(f"We have added {num_added_toks} tokens")

In [ ]:
class ELECTRAForMultiLabelClassification(nn.Module):
    def __init__(self, num_labels, dropout_rate=0.3):
        super(ELECTRAForMultiLabelClassification, self).__init__()
        self.electra = ElectraModel.from_pretrained('google/electra-base-discriminator')
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.electra.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.electra(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

In [ ]:
train_dataset_electra = EmotionDataset(oversampled_texts, oversampled_labels, electra_tokenizer, max_len=128)
val_dataset_electra = EmotionDataset(ds['validation']['text'], ds['validation']['labels'], electra_tokenizer, max_len=128)
test_dataset_electra = EmotionDataset(ds['test']['text'], ds['test']['labels'], electra_tokenizer, max_len=128)

# Create data loaders
train_loader = DataLoader(train_dataset_electra, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset_electra, batch_size=16)
test_loader = DataLoader(test_dataset_electra, batch_size=16)

In [ ]:
def hyperparameter_tuning_electra():
    gammas = [2, 3]
    learning_rates = [5e-5, 1e-5, 7e-6]
    num_epochs = 3
    dropout_rate = 0.5
    weight_decay = 0.1

    best_params = None
    best_val_f1 = 0
    best_model_state = None
    best_history = None
    best_epoch = 0

    results = []

    for gamma in gammas:
        for lr in learning_rates:
            print(f"Training ELECTRA with gamma={gamma}, lr={lr}")
            electra_model = ELECTRAForMultiLabelClassification(num_labels,dropout_rate=dropout_rate).to(device)
            electra_model.electra.resize_token_embeddings(len(electra_tokenizer))

            optimizer = torch.optim.AdamW(electra_model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = FocalLoss(gamma=gamma)
            scaler = GradScaler()

            train_losses = []
            val_losses = []
            val_f1_scores = []

            for epoch in range(num_epochs):
                train_loss = train(electra_model, train_loader, optimizer, criterion, device, scaler)
                val_loss, val_probs, val_labels = evaluate(electra_model, val_loader, criterion, device)
                val_preds = (val_probs > threshold).astype(int)
                val_f1 = f1_score(val_labels, val_preds, average='weighted')

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                val_f1_scores.append(val_f1)
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_params = {'gamma': gamma, 'lr': lr}
                    best_model_state = electra_model.state_dict()
                    best_history = {
                        'train_losses': train_losses.copy(),
                        'val_losses': val_losses.copy(),
                        'val_f1_scores': val_f1_scores.copy()
                    }
                    best_epoch = epoch + 1

            results.append({
                'gamma': gamma,
                'lr': lr,
                'losses': train_losses,
                'val_losses': val_losses,
                'f1_scores': val_f1_scores
            })


    print(f"Best parameters: {best_params}")
    print(f"Best F1 score: {best_val_f1}")
    print(f"Best epoch: {best_epoch}")


    hyperparameter_tuning_plots(results,"ELECTRA",num_epochs)

    torch.save(best_model_state, 'electra_best_model.pth')

    return best_params, best_model_state, best_history, best_epoch

# Run the hyperparameter tuning
print("Running ELECTRA hyperparameter tuning...")
best_params, best_model_state, best_history, best_epoch = hyperparameter_tuning_electra()

In [ ]:

electra_model = ELECTRAForMultiLabelClassification(num_labels, dropout_rate=0.5).to(device)
electra_model.electra.resize_token_embeddings(len(electra_tokenizer))
electra_model.load_state_dict(best_model_state)  # Load the best model state


optimizer = torch.optim.AdamW(electra_model.parameters(), lr=best_params['lr'], weight_decay=0.1)
criterion = FocalLoss(gamma=best_params['gamma'])
scaler = GradScaler()
print("Further model training of the best model found in hyperparameter tuning")
num_epochs = 5
patience = 2
best_val_f1 = max(best_history['val_f1_scores'])
counter = 0


train_losses = best_history['train_losses']
val_losses = best_history['val_losses']
val_f1_scores = best_history['val_f1_scores']

for epoch in range(num_epochs):
    train_loss = train(electra_model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_probs, val_labels = evaluate(electra_model, val_loader, criterion, device)
    val_preds = (val_probs > threshold).astype(int)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1_scores.append(val_f1)

    print(f"Epoch {best_epoch + epoch + 1}/{best_epoch + num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation F1: {val_f1:.4f}")


    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(electra_model.state_dict(), 'electra_best_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience and epoch + 1 < num_epochs :
                print(f"Early stopping triggered at epoch {best_epoch + epoch + 1}")
                break



In [ ]:
total_epochs = best_epoch + epoch + 1
final_model_plots(total_epochs,best_epoch,train_losses,val_losses,val_f1_scores,'ELECTRA')

In [ ]:
electra_model.load_state_dict(torch.load('electra_best_model.pth'))
electra_metrics = evaluate_and_print(electra_model, test_loader, criterion, device, "ELECTRA", threshold=0.4)

In [ ]:
plot_performance_curves(electra_metrics['labels'], electra_metrics['preds'], "ELECTRA")